In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_GetSelectedAssessmentWorklist
# MAGIC Finds selected assessment batches that are eligible for automatic Job 1B onboarding.
# MAGIC Output items contain strictly connection_id and assessment_id.
# MAGIC This notebook performs metadata reads only: no adapter, secret, or JDBC access.

# COMMAND ----------

import json
from pyspark.sql import functions as F

try:
    from src.identifiers import escape_string_literal
    from src.source_identity import require_source_system, canonical_source_system_sql
    from src.worklist_utils import TASK_VALUE_LIMIT_BYTES, validate_task_value_payload
except ModuleNotFoundError:
    from identifiers import escape_string_literal
    from source_identity import require_source_system, canonical_source_system_sql
    from worklist_utils import TASK_VALUE_LIMIT_BYTES, validate_task_value_payload

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("catalog", "da_accelerators")
dbutils.widgets.text("control_schema", "control")
dbutils.widgets.text("max_batches", "0")
dbutils.widgets.text("only_connection_ids", "")
dbutils.widgets.text("only_assessment_ids", "")
dbutils.widgets.dropdown("include_failed_retries", "false", ["true", "false"])

run_id = dbutils.widgets.get("run_id").strip()
source_system_raw = dbutils.widgets.get("source_system").strip()
catalog = dbutils.widgets.get("catalog").strip()
control_schema = dbutils.widgets.get("control_schema").strip()
include_failed_retries_raw = dbutils.widgets.get("include_failed_retries").strip().lower()

if not run_id:
    raise ValueError("run_id is required")
if not source_system_raw:
    raise ValueError("source_system is required")
if not catalog:
    raise ValueError("catalog is required")
if not control_schema:
    raise ValueError("control_schema is required")

source_system = require_source_system(source_system_raw, "NB_GetSelectedAssessmentWorklist")
include_failed_retries = include_failed_retries_raw in ("true", "1", "yes")

try:
    max_batches = int(dbutils.widgets.get("max_batches").strip() or "0")
except ValueError as exc:
    raise ValueError("max_batches must be a non-negative integer") from exc
if max_batches < 0:
    raise ValueError("max_batches must be a non-negative integer")

raw_only_conns = [
    value.strip() for value in
    dbutils.widgets.get("only_connection_ids").split(",") if value.strip()
]
if len(raw_only_conns) != len(set(raw_only_conns)):
    raise ValueError("only_connection_ids contains duplicate values")

raw_only_assessments = [
    value.strip() for value in
    dbutils.widgets.get("only_assessment_ids").split(",") if value.strip()
]
if len(raw_only_assessments) != len(set(raw_only_assessments)):
    raise ValueError("only_assessment_ids contains duplicate values")

def _fqn(table_name):
    parts = (catalog, control_schema, table_name)
    return ".".join("`" + part.replace("`", "``") + "`" for part in parts)

assessments = spark.table(_fqn("source_assessment")).alias("sa")
connections = spark.table(_fqn("source_connection")).alias("sc")

base_joined = (
    assessments.join(
        connections,
        F.col("sa.connection_id") == F.col("sc.connection_id"),
        "inner",
    )
    .filter(F.col("sa.object_type") == F.lit("TABLE"))
    .filter(F.upper(F.trim(F.col("sa.compatibility_status"))).isin(["COMPATIBLE", "REVIEW"]))
    .filter(F.col("sa.is_selected") == F.lit(True))
    .filter(F.col("sc.is_active") == F.lit(True))
    .filter(F.upper(F.trim(F.col("sc.connection_status"))) == F.lit("VALID"))
    .filter(F.col("sc.secret_scope").isNotNull() & (F.trim(F.col("sc.secret_scope")) != ""))
    .filter(F.expr(f"{canonical_source_system_sql('sc.source_system')} = {escape_string_literal(source_system)}"))
    .filter(F.expr(f"{canonical_source_system_sql('sa.source_system')} = {canonical_source_system_sql('sc.source_system')}"))
)

if include_failed_retries:
    base_joined = base_joined.filter(
        (F.col("sa.selection_status").isNull()
         | (F.trim(F.col("sa.selection_status")) == "")
         | F.upper(F.trim(F.col("sa.selection_status"))).isin(["SELECTED", "FAILED"]))
        & (~F.upper(F.trim(F.coalesce(F.col("sa.selection_status"), F.lit("")))).isin(
            ["ONBOARDING", "REGISTERED", "ONBOARDED", "REVIEW_REQUIRED", "BLOCKED"]))
    )
else:
    base_joined = base_joined.filter(
        (F.col("sa.selection_status").isNull()
         | (F.trim(F.col("sa.selection_status")) == "")
         | (F.upper(F.trim(F.col("sa.selection_status"))) == F.lit("SELECTED")))
        & (~F.upper(F.trim(F.coalesce(F.col("sa.selection_status"), F.lit("")))).isin(
            ["ONBOARDING", "REGISTERED", "ONBOARDED", "FAILED", "REVIEW_REQUIRED", "BLOCKED"]))
    )

if raw_only_conns:
    base_joined = base_joined.filter(F.col("sa.connection_id").isin(raw_only_conns))

if raw_only_assessments:
    base_joined = base_joined.filter(F.col("sa.assessment_id").isin(raw_only_assessments))

overlap_check = (
    base_joined
    .groupBy("sa.connection_id", "sa.source_database", "sa.source_schema", "sa.object_name")
    .agg(F.countDistinct("sa.assessment_id").alias("conflicting_assessment_count"))
    .filter(F.col("conflicting_assessment_count") > 1)
)

overlapping_rows = overlap_check.collect()
if overlapping_rows:
    sample = overlapping_rows[0]
    conflict_desc = {
        "connection_id": sample["connection_id"],
        "source_database": sample.get("source_database") or "",
        "source_schema": sample["source_schema"],
        "object_name": sample["object_name"],
        "conflicting_assessment_count": int(sample["conflicting_assessment_count"]),
    }
    raise ValueError(
        f"AMBIGUOUS_SELECTED_ASSESSMENT: Overlapping selected table detected across multiple assessment IDs: "
        f"{json.dumps(conflict_desc)}. Operator must deselect the obsolete assessment row before onboarding."
    )

batches = (
    base_joined
    .select(
        F.col("sa.connection_id").alias("connection_id"),
        F.col("sa.assessment_id").alias("assessment_id"),
    )
    .distinct()
    .orderBy("connection_id", "assessment_id")
)

if max_batches > 0:
    batches = batches.limit(max_batches)

batch_rows = batches.collect()

worklist = [
    {
        "connection_id": row["connection_id"],
        "assessment_id": row["assessment_id"],
    }
    for row in batch_rows
]

for item in worklist:
    if set(item.keys()) != {"connection_id", "assessment_id"}:
        raise ValueError(f"Worklist item contains unexpected keys: {list(item.keys())}")

validate_task_value_payload(worklist, key="worklist", limit_bytes=TASK_VALUE_LIMIT_BYTES)

worklist_count = len(worklist)
if worklist:
    pair_keys = [f"{item['connection_id']}::{item['assessment_id']}" for item in worklist]
    selected_table_count = (
        base_joined
        .filter(
            F.concat_ws("::", F.col("sa.connection_id"), F.col("sa.assessment_id")).isin(pair_keys)
        )
        .count()
    )
    connection_count = len({item["connection_id"] for item in worklist})
else:
    selected_table_count = 0
    connection_count = 0

dbutils.jobs.taskValues.set(key="run_id", value=run_id)
dbutils.jobs.taskValues.set(key="source_system", value=source_system)
dbutils.jobs.taskValues.set(key="worklist", value=worklist)
dbutils.jobs.taskValues.set(key="worklist_count", value=worklist_count)
dbutils.jobs.taskValues.set(key="selected_table_count", value=selected_table_count)
dbutils.jobs.taskValues.set(key="connection_count", value=connection_count)
print(f"Selected assessment worklist: batches={worklist_count}, tables={selected_table_count}, connections={connection_count}")

if worklist_count == 0:
    exit_payload = {
        "status": "SUCCEEDED",
        "business_status": "NO_SELECTED_ASSESSMENTS",
        "run_id": run_id,
        "source_system": source_system,
        "worklist_count": 0,
        "selected_table_count": 0,
        "connection_count": 0,
    }
else:
    exit_payload = {
        "status": "SUCCEEDED",
        "business_status": "READY",
        "run_id": run_id,
        "source_system": source_system,
        "worklist_count": worklist_count,
        "selected_table_count": selected_table_count,
        "connection_count": connection_count,
        "worklist": worklist,
    }

dbutils.notebook.exit(json.dumps(exit_payload))